# Gate 1: readout stability on the base model

Reads `results/manifest.csv` and the `eval_base` / `eval_base_repeat` run folders.
Writes `results/gate1_base_readouts.csv`, `results/gate1_trigger_position.csv` and `reports/figures/2026-11-10-gate1.png`.
Trains nothing.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
man = pd.read_csv(ROOT / "results/manifest.csv")
ev = man[(man.run_type == "eval") & (man.status == "ok")]
def load(prefix):
    ids = ev[ev.run_id.str.contains(f"-{prefix}-es")].run_id
    return pd.concat([pd.read_csv(ROOT / "runs" / r / "metrics.csv") for r in ids], ignore_index=True)
base = load("eval_base").sort_values("eval_seed")
rep = load("eval_base_repeat")
cols = ["run_id", "eval_seed", "target_logprob", "target_logprob_untriggered", "target_rate", "target_rate_untriggered",
        "clean_loss", "target_logprob_per_token", "n_samples", "n_prompts", "n_clean_tokens", "seconds"]
base[cols].to_csv(ROOT / "results/gate1_base_readouts.csv", index=False)
base[cols].round(4)

In [ ]:
summ = base[["target_logprob", "target_logprob_untriggered", "clean_loss", "target_rate", "target_rate_untriggered"]].agg(["median", "min", "max", "std"]).T
summ["range"] = summ["max"] - summ["min"]
summ.round(4)

In [ ]:
# Determinism: the repeat of eval seed 0 must match exactly.
b0 = base[base.eval_seed == 0].iloc[0]; r0 = rep.iloc[0]
num = ["target_logprob", "target_logprob_untriggered", "target_rate", "target_rate_untriggered", "clean_loss"]
det = pd.DataFrame({"eval_base_es0": b0[num].astype(float), "repeat_es0": r0[num].astype(float)})
det["abs_diff"] = (det.eval_base_es0 - det.repeat_es0).abs()
c0 = pd.read_csv(ROOT / "runs" / b0.run_id / "completions.csv"); c1 = pd.read_csv(ROOT / "runs" / r0.run_id / "completions.csv")
print("completions identical:", c0.text.equals(c1.text), "| repeat run:", r0.run_id)
det

In [ ]:
# Per-prompt trigger effect on the base model, by trigger position (pooled over eval seeds).
rows = []
for r in base.run_id:
    d = pd.read_csv(ROOT / "runs" / r / "per_prompt.csv"); d["run_id"] = r
    rows.append(d)
pp = pd.concat(rows, ignore_index=True)
pp["delta"] = pp.target_logprob - pp.target_logprob_untriggered
pp["position"] = np.where(pp.trigger_word_index == pp.n_words, "end of prompt",
                 np.where(pp.trigger_word_index == 1, "after word 1", "middle"))
pos = pp.groupby("position")["delta"].agg(n="size", median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75), frac_positive=lambda x: (x > 0).mean())
pos.to_csv(ROOT / "results/gate1_trigger_position.csv")
pos.round(2)

In [ ]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3), constrained_layout=True)
s = base.eval_seed
a = ax[0]
a.plot(s, base.target_logprob, "o", color=BLUE, ms=6, mec="#fcfcfb", mew=1.5)
a.plot(s, base.target_logprob_untriggered, "s", color=ORANGE, ms=6, mec="#fcfcfb", mew=1.5)
a.annotate("triggered", (s.iloc[-1], base.target_logprob.iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", color=INK)
a.annotate("untriggered", (s.iloc[-1], base.target_logprob_untriggered.iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", color=INK)
a.set_xlim(-0.5, 12.5); a.set_xlabel("evaluation seed"); a.set_ylabel("target_logprob (nats, whole target)")
a.set_title("A  target log-prob, base model", loc="left", color=INK)
a = ax[1]
a.plot(s, base.clean_loss, "o", color=BLUE, ms=6, mec="#fcfcfb", mew=1.5)
a.set_xlabel("evaluation seed"); a.set_ylabel("clean_loss (nats/token)")
a.set_title("B  clean loss, base model", loc="left", color=INK)
a = ax[2]
order = ["after word 1", "middle", "end of prompt"]
rng = np.random.default_rng(0)
for i, p in enumerate(order):
    v = pp.loc[pp.position == p, "delta"].to_numpy()
    a.scatter(i + rng.uniform(-0.25, 0.25, len(v)), v, s=4, color=BLUE, alpha=0.25, linewidths=0)
    a.hlines(np.median(v), i - 0.32, i + 0.32, color=INK, lw=2)
a.axhline(0, color=MUTED, lw=0.8)
a.set_xticks(range(3), [f"{p}\n(n={(pp.position == p).sum()})" for p in order]); a.set_ylabel("triggered − untriggered (nats)")
a.set_title("C  per-prompt trigger effect (all seeds)", loc="left", color=INK)
for a in ax: a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True)
fig.savefig(ROOT / "reports/figures/2026-11-10-gate1.png", dpi=160)
plt.show()